<a href="https://colab.research.google.com/github/otoperalias/Coyuntura/blob/main/clases/Tema2_IV_ap1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Tema 2 IV. Coyuntura en el mercado laboral.

## Apéndice 1: Extracción automática de datos de parados registrados.

Los datos de demandantes de empleo parados (parados registrados) están disponibles en [este enlace](https://www.sepe.es/HomeSepe/que-es-el-sepe/estadisticas/empleo/estadisticas-nuevas.html). Como puede apreciarse, es muy tedioso descargar mes a mes. A continuación procedemos a extraer los datos necesarios directamente en Python.

#### 1. Creamos un bucle para extraer los datos deseados  

Esta es la parte más compleja. A través del bucle se rellenan de manera iterativa tres listas, una donde se guardan las fechas, otra para los parados de Sevilla y una última para los de España.  
Es necesario ver la estructura del url del que leer los archivos excel del Ministerio. Como puede verse, hay variaciones en dicho url en diferentes momentos de tiempo.  
Este bucle tarda mucho en ejecurarse (más de 5 minutos).

In [30]:
import pandas as pd

In [31]:
meses={'enero':"01",'febrero':"02",'marzo':"03",'abril':"04",'mayo':"05",'junio':"06",'julio':"07",'agosto':"08",'septiembre':"09",'octubre':"10",'noviembre':"11",'diciembre':"12"}
years=["2022","2023",'2024','2025','2026'] # "2020","2021",

preg_fecha=[]
preg_sev=[]
preg_esp=[]

for year in years:
  for mes in meses.keys():
    fecha=year+"-"+meses[mes]
    #if fecha=="2020-01" or fecha=="2020-02": # Para estos dos meses la url es ligeramente diferente!
    #  link="https://www.sepe.es/SiteSepe/contenidos/que_es_el_sepe/estadisticas/datos_estadisticos/empleo/datos/"+year+"/"+mes+"_"+year+"/ESTADISTICA_DE_-EMPLEO.xls"
    if int(year)<2024 or (fecha=="2024-01"):
      link="https://www.sepe.es/SiteSepe/contenidos/que_es_el_sepe/estadisticas/datos_estadisticos/empleo/datos/"+year+"/"+mes+"_"+year+"/ESTADISTICA-DE-EMPLEO.xls"
    elif int(year)<2026 or (int(year)==2026 and int(meses[mes])<5): # O sea: cuando el año es >2024
      link="https://www.sepe.es/SiteSepe/contenidos/que_es_el_sepe/estadisticas/datos_estadisticos/empleo/datos/"+year+"/"+mes[0].upper()+mes[1:]+"_"+year+"/ESTADISTICA-DE-EMPLEO.xls"
    elif fecha=="2026-05": # Para este mes también es ligeramente diferente!
      link="https://www.sepe.es/SiteSepe/contenidos/que_es_el_sepe/estadisticas/datos_estadisticos/empleo/datos/"+year+"/"+mes[0].upper()+mes[1:]+"_"+year+"/ESTADISTICA_DE_-EMPLEO.xls0"
    else:
      link="https://www.sepe.es/SiteSepe/contenidos/que_es_el_sepe/estadisticas/datos_estadisticos/empleo/datos/"+year+"/"+mes[0].upper()+mes[1:]+"_"+year+"/ESTADISTICA_DE_EMPLEO.xls"
    preg=pd.read_excel(link,sheet_name="6.2")
    sev=preg.loc[preg["Unnamed: 1"]=="SEVILLA","Unnamed: 2"].iloc[0]
    esp=preg.loc[preg["Unnamed: 1"]=="TOTAL ZONA GEOGRAFICA","Unnamed: 2"].iloc[0]
    preg_fecha.append(fecha)
    preg_sev.append(sev)
    preg_esp.append(esp)
    if fecha=="2026-09":
      break

#### 2. Creamos una dataframe a partir de las listas

In [32]:
diccionario={'fecha':preg_fecha,'sev':preg_sev, 'esp':preg_esp}
parados=pd.DataFrame(diccionario)

#### 3. Convertimos la fecha en formato "datetime" y lo establecemos como index.

In [33]:
parados['fecha']=pd.to_datetime(parados['fecha'])
parados.set_index("fecha", inplace=True)

#### 4. Guardamos como csv.

In [34]:
parados.to_csv("parados_Sevilla.csv")